# Retail Sales Analysis Dashboard
**Tools:** SQLite · pandas · matplotlib · seaborn  
**Dataset:** Synthetic Superstore Sales (10,000 rows, 2021-2023)

## 0. Setup

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Retail Sales Analysis Dashboard
# **Tools:** SQLite · pandas · matplotlib · seaborn
# **Dataset:** Synthetic Superstore Sales (10,000 rows, 2021-2023)

## 0. Setup

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")

# ── Style ─────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

COLORS = {
    "blue":   "#378ADD",
    "teal":   "#1D9E75",
    "coral":  "#D85A30",
    "purple": "#7F77DD",
    "amber":  "#BA7517",
    "gray":   "#888780",
}

DB_PATH = "sales.db"

# Check DB exists
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        "sales.db not found. Please run  python generate_dataset.py  first."
    )

conn = sqlite3.connect(DB_PATH)

def query(sql):
    """Run a SQL string, return a DataFrame."""
    return pd.read_sql_query(sql, conn)

print("Connected to", DB_PATH)

## 1. Overview KPIs

In [ ]:
kpi = query("""
SELECT
    COUNT(DISTINCT order_id)                         AS total_orders,
    ROUND(SUM(sales), 2)                             AS total_revenue,
    ROUND(SUM(profit), 2)                            AS total_profit,
    ROUND(AVG(sales), 2)                             AS avg_order_value,
    ROUND(SUM(profit) / SUM(sales) * 100, 1)         AS profit_margin_pct
FROM sales
""").iloc[0]

print("\n━━━  KEY PERFORMANCE INDICATORS  ━━━")
print(f"  Total Orders      : {int(kpi.total_orders):,}")
print(f"  Total Revenue     : ${kpi.total_revenue:,.2f}")
print(f"  Total Profit      : ${kpi.total_profit:,.2f}")
print(f"  Avg Order Value   : ${kpi.avg_order_value:,.2f}")
print(f"  Profit Margin     : {kpi.profit_margin_pct}%")

# ── KPI card chart ────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 2.8))
fig.suptitle("Key Performance Indicators", fontsize=14, fontweight="bold", y=1.02)

metrics = [
    ("Total Revenue",   f"${kpi.total_revenue/1e6:.2f}M", COLORS["blue"]),
    ("Total Profit",    f"${kpi.total_profit/1e6:.2f}M",  COLORS["teal"]),
    ("Avg Order Value", f"${kpi.avg_order_value:,.0f}",   COLORS["purple"]),
    ("Profit Margin",   f"{kpi.profit_margin_pct}%",       COLORS["amber"]),
]

for ax, (label, value, color) in zip(axes, metrics):
    ax.set_facecolor("#F8F8F8")
    ax.text(0.5, 0.58, value, ha="center", va="center", fontsize=22,
            fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha="center", va="center", fontsize=11,
            color="#666", transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor("#E0E0E0")

plt.tight_layout()
plt.savefig("charts/01_kpi_cards.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/01_kpi_cards.png")

## 2. Monthly Revenue & Profit Trend

In [ ]:
monthly = query("""
SELECT
    STRFTIME('%Y-%m', order_date) AS month,
    ROUND(SUM(sales), 2)          AS revenue,
    ROUND(SUM(profit), 2)         AS profit
FROM sales
GROUP BY month
ORDER BY month
""")

monthly["month_dt"] = pd.to_datetime(monthly["month"])

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(monthly["month_dt"], monthly["revenue"], color=COLORS["blue"],
        linewidth=2.2, label="Revenue", marker="o", markersize=3)
ax.fill_between(monthly["month_dt"], monthly["revenue"],
                alpha=0.08, color=COLORS["blue"])
ax.plot(monthly["month_dt"], monthly["profit"], color=COLORS["teal"],
        linewidth=2, label="Profit", linestyle="--", marker="o", markersize=3)

ax.set_title("Monthly Revenue & Profit Trend (2021–2023)", fontsize=13, pad=12)
ax.set_xlabel("Month"); ax.set_ylabel("USD ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("charts/02_monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/02_monthly_trend.png")

## 3. Revenue by Region

In [ ]:
region_df = query("""
SELECT
    region,
    ROUND(SUM(sales), 2)   AS revenue,
    ROUND(SUM(profit), 2)  AS profit
FROM sales
GROUP BY region
ORDER BY revenue DESC
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar chart
bars = ax1.bar(region_df["region"], region_df["revenue"],
               color=[COLORS["blue"], COLORS["purple"], COLORS["teal"], COLORS["amber"]])
ax1.set_title("Revenue by Region", fontsize=13, pad=10)
ax1.set_ylabel("Revenue (USD)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 500,
             f"${h/1000:.0f}k", ha="center", va="bottom", fontsize=10)

# Pie chart – profit share
ax2.pie(region_df["profit"], labels=region_df["region"],
        colors=[COLORS["blue"], COLORS["purple"], COLORS["teal"], COLORS["amber"]],
        autopct="%1.1f%%", startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax2.set_title("Profit Share by Region", fontsize=13, pad=10)

plt.tight_layout()
plt.savefig("charts/03_region_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/03_region_analysis.png")

## 4. Category & Sub-Category Performance

In [ ]:
cat_df = query("""
SELECT
    category,
    sub_category,
    ROUND(SUM(sales), 2)  AS revenue,
    ROUND(SUM(profit), 2) AS profit
FROM sales
GROUP BY category, sub_category
ORDER BY category, revenue DESC
""")

cat_total = cat_df.groupby("category")["revenue"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cat_colors = {"Technology": COLORS["blue"],
              "Furniture":  COLORS["coral"],
              "Office Supplies": COLORS["teal"]}

for ax, cat in zip(axes, cat_total.index):
    sub = cat_df[cat_df["category"] == cat].sort_values("revenue")
    bars = ax.barh(sub["sub_category"], sub["revenue"],
                   color=cat_colors[cat], alpha=0.85)
    ax.set_title(cat, fontsize=12, fontweight="bold", color=cat_colors[cat])
    ax.set_xlabel("Revenue (USD)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 50, bar.get_y() + bar.get_height()/2,
                f"${w/1000:.1f}k", va="center", fontsize=9)

fig.suptitle("Revenue by Category & Sub-Category", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("charts/04_category_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/04_category_breakdown.png")

## 5. Top 10 Products by Revenue

In [ ]:
top10 = query("""
SELECT
    product_name,
    sub_category,
    ROUND(SUM(sales), 2)  AS revenue,
    ROUND(SUM(profit), 2) AS profit
FROM sales
GROUP BY product_name, sub_category
ORDER BY revenue DESC
LIMIT 10
""")

fig, ax = plt.subplots(figsize=(11, 5))
palette = sns.color_palette("Blues_d", len(top10))
bars = ax.barh(top10["product_name"], top10["revenue"], color=palette)
ax.set_title("Top 10 Products by Revenue", fontsize=13, pad=10)
ax.set_xlabel("Total Revenue (USD)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.invert_yaxis()
for bar in bars:
    w = bar.get_width()
    ax.text(w + 100, bar.get_y() + bar.get_height()/2,
            f"${w:,.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("charts/05_top10_products.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/05_top10_products.png")

## 6. Discount Impact on Profit Margin

In [ ]:
disc_df = query("""
SELECT
    CASE
        WHEN discount = 0    THEN '0%'
        WHEN discount <= 0.1 THEN '10%'
        WHEN discount <= 0.2 THEN '20%'
        WHEN discount <= 0.3 THEN '30%'
        ELSE                      '40%+'
    END                                   AS discount_band,
    COUNT(*)                              AS orders,
    ROUND(AVG(profit / sales * 100), 1)   AS avg_margin_pct
FROM sales
GROUP BY discount_band
ORDER BY discount_band
""")

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = [COLORS["teal"] if m > 0 else COLORS["coral"]
              for m in disc_df["avg_margin_pct"]]
bars = ax.bar(disc_df["discount_band"], disc_df["avg_margin_pct"], color=bar_colors)
ax.axhline(0, color="#888", linewidth=0.8, linestyle="--")
ax.set_title("Average Profit Margin by Discount Level", fontsize=13, pad=10)
ax.set_xlabel("Discount Band"); ax.set_ylabel("Avg Margin (%)")
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2,
            h + (0.3 if h >= 0 else -0.8),
            f"{h:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("charts/06_discount_impact.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/06_discount_impact.png")

## 7. Shipping Mode Distribution

In [ ]:
ship_df = query("""
SELECT
    ship_mode,
    COUNT(*) AS orders,
    ROUND(SUM(sales), 2) AS revenue
FROM sales
GROUP BY ship_mode
ORDER BY orders DESC
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ship_colors = [COLORS["blue"], COLORS["teal"], COLORS["purple"], COLORS["amber"]]

ax1.pie(ship_df["orders"], labels=ship_df["ship_mode"], autopct="%1.1f%%",
        colors=ship_colors, startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax1.set_title("Orders by Ship Mode", fontsize=13)

ax2.bar(ship_df["ship_mode"], ship_df["revenue"], color=ship_colors)
ax2.set_title("Revenue by Ship Mode", fontsize=13)
ax2.set_ylabel("Revenue (USD)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
plt.xticks(rotation=15)

plt.tight_layout()
plt.savefig("charts/07_shipping_modes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/07_shipping_modes.png")

## 8. Correlation Heatmap

In [ ]:
num_df = query("SELECT sales, profit, quantity, discount FROM sales")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(num_df.corr(), annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Heatmap – Numeric Features", fontsize=13, pad=10)
plt.tight_layout()
plt.savefig("charts/08_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → charts/08_correlation_heatmap.png")

## 9. Year-over-Year Summary Table

In [ ]:
yoy = query("""
SELECT
    STRFTIME('%Y', order_date)       AS year,
    COUNT(DISTINCT order_id)         AS orders,
    ROUND(SUM(sales), 2)             AS revenue,
    ROUND(SUM(profit), 2)            AS profit,
    ROUND(SUM(profit)/SUM(sales)*100,1) AS margin_pct
FROM sales
GROUP BY year
ORDER BY year
""")

yoy["revenue_fmt"] = yoy["revenue"].apply(lambda x: f"${x:,.0f}")
yoy["profit_fmt"]  = yoy["profit"].apply(lambda x: f"${x:,.0f}")
yoy["margin_fmt"]  = yoy["margin_pct"].apply(lambda x: f"{x}%")

print("\n━━━  YEAR-OVER-YEAR SUMMARY  ━━━")
print(yoy[["year","orders","revenue_fmt","profit_fmt","margin_fmt"]]
      .rename(columns={"year":"Year","orders":"Orders",
                       "revenue_fmt":"Revenue","profit_fmt":"Profit",
                       "margin_fmt":"Margin"}).to_string(index=False))

## 10. Wrap-up

In [ ]:
conn.close()
print("\n✅  Analysis complete! All charts saved to charts/")
print("    See queries.sql for the raw SQL used in each section.")